In [ ]:
# Enviorment: Google Colab
# install dependencies
pip install langchain_community langchain_experimental langchain_groq

##1. Character Splitting.
 - What it is: Splits text based purely on character count.

 - How it works: You set a max length (say 30 characters). The text is sliced sequentially into chunks of that size.

 - Pros: Simple, fast, language-agnostic.

 - Cons: Cuts may occur mid-sentence or mid-word → losing semantic meaning.

 - Use case: Quick experiments, very uniform datasets.



In [ ]:
from langchain.text_splitter import CharacterTextSplitter

text = "Artificial Intelligence is transforming industries. It powers chatbots, autonomous cars, and medical imaging."

splitter = CharacterTextSplitter(
    chunk_size=30,
    chunk_overlap=0,
    separator=""   # important! allows true character-level split
)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: {chunk}")


Chunk 1: Artificial Intelligence is tra
Chunk 2: nsforming industries. It power
Chunk 3: s chatbots, autonomous cars, a
Chunk 4: nd medical imaging.


##2. Recursive Character Splitting

 - What it is: A smarter version of character splitting.

 - How it works: It tries to split along natural boundaries:

    - First, try to split by paragraphs.

    - If a chunk is still too big, split by sentences.

    - If still too big, split by words.

    - Finally, fallback to raw characters.

 - Pros: Preserves semantic context better than naive character splitting.

 - Cons: Still doesn’t understand meaning—just follows hierarchy.

 - Use case: Most common in RAG pipelines when you want balance between chunk size and context.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text = "Artificial Intelligence is transforming industries. It powers chatbots, autonomous cars, and medical imaging."

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=0,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: {chunk}")

Chunk 1: Artificial Intelligence is transforming industries
Chunk 2: . It powers chatbots, autonomous cars, and medical
Chunk 3: imaging.


##3. Document-Specific Splitting

 - What it is: Custom splitting rules depending on the document structure.

 - How it works: Instead of blind splitting, you parse structured docs like:

    - Markdown → split by headers.

    - PDFs → split by sections/tables.

    - Legal contracts → split by clauses.

    - Code → split by functions/classes.

- Pros: Preserves logical units.

- Cons: Requires document-specific parsing logic.

- Use case: Research papers, legal docs, financial reports, codebases.

In [ ]:
text = """Clause 1: This agreement is valid for 2 years.
Clause 2: The tenant shall pay rent by the 5th of each month.
Clause 3: The landlord is responsible for maintenance."""

# Custom splitter by "Clause"
chunks = text.split("Clause ")
chunks = [f"Clause {c.strip()}" for c in chunks if c.strip()]

for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: {chunk}")

Chunk 1: Clause 1: This agreement is valid for 2 years.
Chunk 2: Clause 2: The tenant shall pay rent by the 5th of each month.
Chunk 3: Clause 3: The landlord is responsible for maintenance.


##4. Semantic Splitting

* What it is: Splits text based on meaning rather than length.

* How it works: Uses embeddings (vector representations) or LLMs to detect topic shifts and semantic coherence:

  * If similarity between two sentences drops below a threshold → new chunk.

  * Keeps contextually related sentences together.

* Pros: Produces chunks that align with human understanding.

* Cons: Computationally more expensive, relies on good embeddings/LLMs.

* Use case: Long-form text (books, transcripts, articles) where coherence matters.

In [4]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings  # for HF embeddings

text = """Artificial Intelligence is transforming industries.
It powers chatbots, autonomous cars, and medical imaging.
Meanwhile, climate change remains a global challenge.
Governments are working on sustainability initiatives."""

# Initialize Hugging Face embeddings (using a popular sentence transformer)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Use SemanticChunker with HF embeddings
splitter = SemanticChunker(embeddings)

# Split the text into semantically meaningful chunks
chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: {chunk}")

/tmp/ipython-input-534286434.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunk 1: Artificial Intelligence is transforming industries. It powers chatbots, autonomous cars, and medical imaging. Meanwhile, climate change remains a global challenge.
Chunk 2: Governments are working on sustainability initiatives.


##5. Agentic Splitting
###langchain agentic splitter
While LangChain does not have a module explicitly called an "agentic splitter," the term describes an advanced, semantic-based strategy for chunking documents. This technique uses a large language model (LLM) or embedding model to guide the splitting process, ensuring that the resulting document chunks are semantically coherent and contextually relevant, rather than simply breaking text by character or token count. <br>

This contrasts with LangChain's traditional text splitters, which rely on rules-based or structural methods. The "agentic" approach adds intelligence to the chunking process, improving the performance of retrieval-augmented generation (RAG) systems. <br>

###How an agentic splitter works
The process for a semantic or agentic splitter involves several intelligent steps:<br>

1. Split into small units: The document is first broken down into a base unit, typically individual sentences.

2. Generate embeddings: An embedding model is used to create a vector representation (an embedding) for each sentence or small group of sentences.

3. Detect breakpoints: The model then calculates the distance between the embeddings of adjacent sentences or sentence groups using a metric like cosine similarity. Breakpoints are identified wherever the semantic similarity drops below a certain threshold.
4. Combine coherent chunks: Sentences with high semantic similarity are grouped together into larger, more meaningful chunks, ensuring that a single topic is not fragmented.



In [6]:
from langchain_groq import ChatGroq
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from google.colab import userdata


text = """Section 1: Introduction to Python.
Section 2: Data Structures in Python.
Section 3: Machine Learning with Python.
Section 4: Deep Learning with Python."""

# Initialize Hugging Face embeddings (using a popular sentence transformer)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


# Initialize your LLM (like before)
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", api_key=userdata.get("GROQ_API_KEY"))

# Use SemanticChunker with embeddings
splitter = SemanticChunker(embeddings)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: {chunk}")

Chunk 1: Section 1: Introduction to Python.
Chunk 2: Section 2: Data Structures in Python. Section 3: Machine Learning with Python. Section 4: Deep Learning with Python.
